# Python Generators & yield: Huge Data, Tiny Memory

Why does your Python script freeze on big files? In this lesson you'll see exactly how generators and the yield keyword let you process millions of rows using a few hundred bytes of memory, traced step by step with real measurements.

Part of the **Advanced Python** series on NeuralNest.

Run the code cells from top to bottom. Change the numbers and run them again: that's the fastest way to really understand it.

## Why this matters

In my first job, I loaded a two gigabyte server log into a Python list, just to count the errors. My laptop froze. The fix was one keyword: yield. By the end of this video, you'll know exactly what a generator is, how yield pauses a function, and how to process millions of rows while using almost no memory. If you know how a for loop works, you're ready.

## The problem: lists build everything

Let's start with the problem. This list comprehension squares every number from zero up to one million. Python builds the whole list right away, all one million results, before you use a single one. sys dot get size of says the list takes eight million, four hundred forty eight thousand bytes. That's about eight and a half megabytes. And here's a trap that catches experienced developers: get size of only measures the list itself, the slots that point to the numbers. The number objects live somewhere else. Count them too, and it's closer to forty megabytes. Now imagine a log file with a hundred million lines.

In [ ]:
import sys

squares = [n * n for n in range(1_000_000)]
print(len(squares))            # -> 1000000
print(sys.getsizeof(squares))  # -> 8448728

## Memory as the data grows

Here's what that looks like as the data grows. Across the bottom, the number of items, in thousands. Up the side, memory in megabytes. Watch the list line climb: about two megabytes at two hundred fifty thousand items, and eight and a half at one million. It grows in a straight line with your data. The second line is a generator doing the same work. It stays flat, at two hundred bytes, whether it produces ten items or ten million. Let's see how that's possible.

In [ ]:
import matplotlib.pyplot as plt

x = [0.0, 250.0, 500.0, 750.0, 1000.0]
plt.plot(x, [0.0, 2.06, 4.17, 6.68, 8.45], marker='o', label='list')
plt.plot(x, [0.0002, 0.0002, 0.0002, 0.0002, 0.0002], marker='o', label='generator')
plt.xlabel('items (thousands)')
plt.ylabel('memory (MB)')
plt.legend()
plt.show()

## Cook all rotis, or one?

- List: cook every roti first
- Generator: cook one when asked
- Same rotis, far less counter space

Think of a kitchen making rotis for a wedding. The list way: cook every single roti before the first guest sits down. You need a huge counter to stack them all. The generator way: cook one roti each time a guest asks for the next one. The guests get exactly the same rotis, but the counter only ever holds one. This is called lazy evaluation: compute each value only when it's needed. One place the analogy breaks: a real kitchen can cook a second round. A finished generator can't, and we'll see why that matters.

## Generator expression: same work, lazily

Here's the generator version. The only change is the brackets: round brackets instead of square ones. That makes a generator expression. Python doesn't compute any squares yet. It only remembers how to make them, so get size of says two hundred bytes. Then sum pulls the values out one at a time, adds each one to a running total, and lets it go. The answer is the same, about three hundred thirty three quadrillion, but at no moment did a million numbers sit in memory together. These byte counts come from Python three point thirteen; they vary a little between versions.

In [ ]:
import sys

squares = (n * n for n in range(1_000_000))
print(sys.getsizeof(squares))  # -> 200
print(sum(squares))  # -> 333332833333500000

## Writing your own with yield

Generator expressions are great for one-liners. For real logic, you write a generator function. It looks like a normal function, but it uses yield instead of return. Calling count up to with three does not run the body at all. Look at the first print: you get a generator object. The body starts only when you call next. It runs until it reaches yield, hands back one, and pauses right there, with n still in memory. The next call resumes after the yield, adds one, loops, and yields two. Then three. The last line asks for whatever is left, and gets an empty list. So, a question for you: what do you think happens on a fourth call to next?

In [ ]:
def count_up_to(limit):
    n = 1
    while n <= limit:
        yield n
        n += 1

counter = count_up_to(3)
print(counter)        # -> <generator object ...>
print(next(counter))  # -> 1
print(next(counter))  # -> 2
print(next(counter))  # -> 3
print(list(counter))  # -> []

## Tracing every next() call

**count_up_to(3)** (highlighted rows in **bold**)

| call | runs until | returns |
|---|---|---|
| next #1 | first yield | 1 |
| next #2 | yield again | 2 |
| next #3 | yield again | 3 |
| **next #4** | **loop ends** | **StopIteration** |

Let's trace it call by call. The first next runs from the top of the function down to the first yield, and returns one. The second resumes right after that yield, adds one to n, loops back, and returns two. The third returns three. Now the answer to the question. On the fourth call, n becomes four, the while condition fails, and the function ends. When a generator ends, next raises an exception called Stop Iteration. That's the highlighted row. You rarely see it, because a for loop catches it for you and simply stops.

## The life of a generator

1. Call function
2. Get generator
3. Run to yield
4. Pause, keep state
5. Resume on next
6. StopIteration

*Paused between values, never restarted*

Here's the whole life cycle. Step one: you call the function. Step two: instead of running, it hands you a generator object. Step three: next runs the body until the next yield. Step four: the function pauses, and keeps all its local variables, exactly where it stopped. Step five: the next call resumes from that point, not from the top. And step six: when the body finishes, it raises Stop Iteration, and the generator is done for good. So far, we've seen why lists use memory and how yield pauses a function. Now let's use it on real data.

## My log file, fixed with generators

Back to my frozen laptop. Here's a smaller version of the fix. fake log stands in for the real file: it yields one log line at a time, and every fourth line is an error. errors is a second generator. It takes any stream of lines and yields only the ones that start with ERROR. Then we chain them: errors, wrapped around fake log of one million. Still, nothing has run. The first next pulls lines through both generators until it finds the first error: ERROR request zero. Then sum adds one for every error that's left: two hundred forty nine thousand, nine hundred ninety nine. With a real file, fake log becomes a function that opens the file and yields each line. The pattern stays the same.

In [ ]:
def fake_log(n):
    for i in range(n):
        level = "ERROR" if i % 4 == 0 else "INFO"
        yield f"{level} request {i}"

def errors(lines):
    for line in lines:
        if line.startswith("ERROR"):
            yield line

found = errors(fake_log(1_000_000))
print(next(found))            # -> ERROR request 0
print(sum(1 for _ in found))  # -> 249999

## A pipeline, one line at a time

1. Read a line
2. Keep errors
3. Count it

*Only one line in memory at a time*

This is called a generator pipeline. Each stage is a generator that pulls from the stage before it. One line is read, checked by the filter, and counted, and only then is the next line read. At any moment, only one line is in memory, whether the file has a thousand lines or a hundred million. That's why the same code works on a two gigabyte log. You can add more stages, like parsing dates or reformatting, and memory stays flat.

## List or generator?

**When to use which**

| feature | list | generator |
|---|---|---|
| memory | every item | one at a time |
| len() and [i] | yes | no |
| loop twice | yes | no, runs once |
| best for | small, reused data | big or endless streams |

So should you use generators everywhere? No. Here's the honest comparison. A list holds every item, so it uses more memory, but you can ask for its length, jump to any index, and loop over it as many times as you like. A generator holds one item at a time, but it has no length, no indexing, and it runs only once. Use a list for small data you'll reuse. Use a generator for big data, data that arrives as a stream, or data that never ends, like readings from a sensor. For a few hundred items, a list is simpler and often just as fast.

## The classic mistake: looping twice

Here's the most common generator bug. We make a generator of four squares and turn it into a list: zero, one, four, nine. Then we do it again, and get an empty list. No error, no warning, just nothing. The first list call used up every value, and a finished generator stays finished. This bites people when they pass one generator to two different functions, like one that counts rows and another that saves them. The second function silently sees no data.

In [ ]:
squares = (n * n for n in range(4))
print(list(squares))   # -> [0, 1, 4, 9]
print(list(squares))   # -> []

## How to avoid the empty second loop

- Need two passes? Use a list
- Or create a fresh generator
- Generators are single-use streams

There are two clean fixes. If you need the data more than once and it fits in memory, make it a list once, and reuse the list. If it doesn't fit, create a fresh generator for each pass, for example by calling the generator function again, which reopens the file from the start. The rule to remember: a generator is a stream, not a container. You read it once, from start to end.

## Recap

- Lists build every value up front
- yield pauses and keeps local state
- Pipelines stream one item at a time
- Generators run once: recreate to reuse

Let's recap. A list builds every value up front, so its memory grows with your data. yield turns a function into a generator that pauses and keeps its local state between values. Chaining generators gives you a pipeline that streams one item at a time, so even a huge file uses almost no memory. And a generator runs once: recreate it, or use a list, when you need the data again. Next in the series: yield from and send, for even more powerful generator pipelines.